1.从环境变量读取DeepSeek apiKey

In [ ]:
import os
from dotenv import load_dotenv


load_dotenv()
# 从环境变量获取 DeepSeek API Key
api_key = os.getenv("DEEPSEEK_API_KEY")
print("DeepSeekApiKey:", api_key)

2.加载 mfd.md 文件。解析出来主要部分的内容

In [ ]:
# from glob import glob
# text_lines = []
# for file_path in glob("./mfd.md", recursive=True):
#     with open(file_path, "r", encoding="utf-8") as file:#encoding="utf-8" 中文使用的GKB会出现编码问题
#         file_text = file.read()
#     text_lines += file_text.split("# ")
# print(text_lines[:1])
# len(text_lines)

import re

with open("./mfd.md", "r", encoding="utf-8") as f:
    file_text = f.read()

# 按 **第xxx条** 拆分，每条法律条文作为一个独立文本块
# 用正则在每个 **第 前面切割，保留分隔符
raw_parts = re.split(r'(?=\*\*第)', file_text)

# 过滤掉空白和非条文内容（如标题行），只保留以 **第 开头的条文
text_lines = [part.strip() for part in raw_parts if part.strip().startswith('**第')]

print(f'共拆分出 {len(text_lines)} 条法律条文')
print(f'示例: {text_lines[0][:80]}...')

3.准备 LLM 和 Embedding 模型

In [ ]:
from openai import OpenAI
from pymilvus import model

#deepseek
deepseek_client = OpenAI(
    api_key=api_key,
    base_url="https://api.deepseek.com/v1",
)

embedding_model = model.dense.SentenceTransformerEmbeddingFunction(
    model_name='BAAI/bge-small-zh-v1.5',
    device='cpu'
)
# all-MiniLM-L6-v2
# embedding_model = model.dense.SentenceTransformerEmbeddingFunction(
#     model_name='all-MiniLM-L6-v2',
#     device='cpu'
# )

4.将数据加载到 Milvus

In [ ]:
#4_1.创建 MilvusClient
from pymilvus import MilvusClient

# milvus_client = MilvusClient(uri="./milvus_demo.db")
milvus_client = MilvusClient(uri="http://localhost:19530")

collection_name = "mfd_collection"

In [ ]:
if milvus_client.has_collection(collection_name):
    milvus_client.drop_collection(collection_name)

In [ ]:
#定义collection
milvus_client.create_collection(
    collection_name=collection_name,
    dimension=512,
    metric_type="IP",
    consistency_level="Strong",
)

In [ ]:
#插入数据
from tqdm import tqdm

data = []

doc_embeddings = embedding_model.encode_documents(text_lines)

for i, line in enumerate(tqdm(text_lines, desc="Creating embeddings")):
    data.append({"id": i, "vector": doc_embeddings[i], "text": line})

milvus_client.insert(collection_name=collection_name, data=data)

In [ ]:
# 查看 mfd_collection 中的数据条数和内容
results = milvus_client.query(
    collection_name="mfd_collection",
    filter="id >= 0",
    output_fields=["id", "text"],
    limit=1000
)

print(f"共 {len(results)} 条数据\n")
for r in results[:5]:  # 先看前5条
    print(f"[id={r['id']}] {r['text'][:80]}...")
    print()


5.构建RAG

In [ ]:
# question = "中华人民共和国民法典第二百零五条是什么?"

# search_res = milvus_client.search(
#     collection_name=collection_name,
#     data=embedding_model.encode_queries(
#         [question]
#     ),  # 将问题转换为嵌入向量
#     limit=3,  # 返回前3个结果
#     search_params={"metric_type": "IP", "params": {}},  # 内积距离
#     output_fields=["text"],  # 返回 text 字段
# )

#embedding 模型做的是语义相似度匹配，不是关键词/编号精确匹配
import re
import json

# === 阿拉伯数字 转 中文数字 ===
def num_to_chinese(num):
    """将阿拉伯数字转为中文数字，如 205 -> 二百零五"""
    digits = '零一二三四五六七八九'
    units = ['', '十', '百', '千', '万']
    n = int(num)
    if n == 0:
        return '零'
    parts = []
    s = str(n)
    length = len(s)
    for i, ch in enumerate(s):
        d = int(ch)
        pos = length - 1 - i  # 个位=0, 十位=1, 百位=2...
        if d == 0:
            # 避免连续零和末尾零
            if parts and parts[-1] != '零':
                parts.append('零')
        else:
            parts.append(digits[d] + units[pos])
    # 去掉末尾的零
    result = ''.join(parts).rstrip('零')
    return result

def extract_article_keywords(question):
    """从问题中提取条文编号，支持中文数字和阿拉伯数字，返回所有可能的关键词列表"""
    keywords = []
    # 匹配中文数字条文号: 第二百零五条
    cn_match = re.search(r'第[零一二三四五六七八九十百千万]+条', question)
    if cn_match:
        keywords.append(cn_match.group())
    # 匹配阿拉伯数字条文号: 第205条
    ar_match = re.search(r'第(\d+)条', question)
    if ar_match:
        num = ar_match.group(1)
        cn_num = num_to_chinese(num)
        keywords.append(f'第{cn_num}条')
    return list(set(keywords))  # 去重

# === 搜索逻辑 ===

question = "中华人民共和国民法典第二百零五条是什么?"
# question = "中华人民共和国民法典第205条是什么?"

keywords = extract_article_keywords(question)
print(f'提取到的条文关键词: {keywords}')

exact_matches = []
if keywords:
    for kw in keywords:
        exact_matches += [line for line in text_lines if kw in line]
    exact_matches = list(set(exact_matches))  # 去重

if exact_matches:
    print(f'精确匹配到 {len(exact_matches)} 条结果:')
    for m in exact_matches:
        print(m[:200])
        print()
    retrieved_lines_with_distances = [(m, 1.0) for m in exact_matches[:3]]
else:
    print('未找到精确匹配，回退到向量搜索')
    search_res = milvus_client.search(
        collection_name=collection_name,
        data=embedding_model.encode_queries([question]),
        limit=3,
        search_params={'metric_type': 'IP', 'params': {}},
        output_fields=['text'],
    )
    retrieved_lines_with_distances = [
        (res['entity']['text'], res['distance']) for res in search_res[0]
    ]

#输出搜索结果
print(json.dumps(retrieved_lines_with_distances, indent=4, ensure_ascii=False))

In [ ]:
# #格式化搜索结果
# import json

# retrieved_lines_with_distances = [
#     (res["entity"]["text"], res["distance"]) for res in search_res[0]
# ]
# print(json.dumps(retrieved_lines_with_distances, indent=4, ensure_ascii=False))

In [ ]:
#检索到的文档转换为字符串格式
context = "\n".join(
    [line_with_distance[0] for line_with_distance in retrieved_lines_with_distances]
)

6.将 RAG 返回的关键数据作为上下文 提供给 LLM

In [ ]:
#LLM提示词
SYSTEM_PROMPT = """
Human: 你是一个 AI 助手。你能够从提供的上下文段落片段中找到问题的答案。
"""
USER_PROMPT = f"""
请使用以下用 <context> 标签括起来的信息片段来回答用 <question> 标签括起来的问题。
<context>
{context}
</context>
<question>
{question}
</question>
"""

In [ ]:
#生成响应
response = deepseek_client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT},
    ],
)
print(response.choices[0].message.content)